# MedlinePlus RAG Index 

**Goal:** build the vector retrieval index (Gemini embeddings + ChromaDB) over the MedlinePlus lay-language glossary and authoritative lay-health guidelines.

**the two sources being used:**
- **Glossary** → MedlinePlus's "Definitions of Health Terms" XML files (small, curated term → definition pairs: fitness, general health, minerals, nutrition, vitamins).
- **Guidelines** → MedlinePlus's Compressed Health Topic XML (daily-updated, full plain-language summaries for every health topic).


i had to run this before just so that the chromadb was configured

```
pip install google-genai chromadb requests python-dotenv
```

**side note just for other people:** Make sure `GEMINI_API_KEY` is set in your `.env` file too

In [1]:
import sys
print(sys.executable)

c:\Users\puria\Documents\Google-2A-faithfulmed-a-multi-agent-system-for-faithful-patient-friendly-medical-explanations\venv\Scripts\python.exe


In [2]:
import os
import re
import time
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import date, timedelta

import requests
import chromadb
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

DATA_DIR = Path('../data/medlineplus')
CHROMA_DIR = '../chroma/medlineplus'  # already covered by .gitignore's chroma/ rule
COLLECTION_NAME = 'medlineplus_glossary_and_guidelines'
EMBED_MODEL = 'gemini-embedding-001'
EMBED_BATCH_SIZE = 80
CHUNK_SIZE_CHARS = 2000
CHUNK_OVERLAP_CHARS = 200

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
print('Client ready.')

Client ready.


## downloading the raw MedlinePlus files

Pulls the five glossary/definitions files plus today's compressed health-topic XML, and unzips into `data/medlineplus/`...but that folder is lowkey already being git-ignored, so nothing large gets committed.

In [3]:
DEFINITION_FILES = {
    'fitness': 'https://medlineplus.gov/xml/fitnessdefinitions.xml',
    'general_health': 'https://medlineplus.gov/xml/generalhealthdefinitions.xml',
    'minerals': 'https://medlineplus.gov/xml/mineralsdefinitions.xml',
    'nutrition': 'https://medlineplus.gov/xml/nutritiondefinitions.xml',
    'vitamins': 'https://medlineplus.gov/xml/vitaminsdefinitions.xml',
}


def download_file(url: str, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'  downloading {url}')
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    out_path.write_bytes(resp.content)
    print(f'  saved to {out_path} ({len(resp.content) / 1024:.0f} KB)')


def guess_todays_topic_zip_url() -> str:
    """MedlinePlus publishes a new compressed health-topic XML most weekdays.
    Try today, then walk backwards a few days if today's isn't up yet."""
    for days_back in range(5):
        d = date.today() - timedelta(days=days_back)
        url = f'https://medlineplus.gov/xml/mplus_topics_compressed_{d.isoformat()}.zip'
        resp = requests.head(url, timeout=10)
        if resp.status_code == 200:
            return url
    raise RuntimeError(
        'Could not auto-find a recent file. Check https://medlineplus.gov/xml.html '
        'and copy the newest "MedlinePlus Compressed Health Topic XML" link manually.'
    )


DATA_DIR.mkdir(parents=True, exist_ok=True)

print('Downloading glossary (Definitions of Health Terms)...')
for name, url in DEFINITION_FILES.items():
    download_file(url, DATA_DIR / f'definitions_{name}.xml')

print('\nDownloading compressed health topic XML (guidelines)...')
zip_url = guess_todays_topic_zip_url()
zip_path = DATA_DIR / 'health_topics.zip'
download_file(zip_url, zip_path)

print('  unzipping...')
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(DATA_DIR)

print('\nDone. Files in data/medlineplus/:')
for f in sorted(DATA_DIR.glob('*')):
    print(' -', f.name)

  downloading https://medlineplus.gov/xml/fitnessdefinitions.xml
  saved to ..\data\medlineplus\definitions_fitness.xml (7 KB)
  downloading https://medlineplus.gov/xml/generalhealthdefinitions.xml
  saved to ..\data\medlineplus\definitions_general_health.xml (7 KB)
  downloading https://medlineplus.gov/xml/mineralsdefinitions.xml
  saved to ..\data\medlineplus\definitions_minerals.xml (8 KB)
  downloading https://medlineplus.gov/xml/nutritiondefinitions.xml
  saved to ..\data\medlineplus\definitions_nutrition.xml (14 KB)
  downloading https://medlineplus.gov/xml/vitaminsdefinitions.xml
  saved to ..\data\medlineplus\definitions_vitamins.xml (9 KB)

  downloading https://medlineplus.gov/xml/mplus_topics_compressed_2026-10-03.zip
  saved to ..\data\medlineplus\health_topics.zip (4670 KB)
  unzipping...

Done. Files in data/medlineplus/:
 - definitions_fitness.xml
 - definitions_general_health.xml
 - definitions_minerals.xml
 - definitions_nutrition.xml
 - definitions_vitamins.xml
 - hea

## checking the XML file setup..

In [4]:
definition_files = sorted(DATA_DIR.glob('definitions_*.xml'))
assert definition_files, 'No definitions_*.xml files found -- run Section 1 first.'

sample_path = definition_files[0]
tree = ET.parse(sample_path)
root = tree.getroot()

print(f'File: {sample_path.name}')
print(f'Root tag: <{root.tag}>')
print(f"Root's direct children tags: {[c.tag for c in list(root)[:5]]}")

if len(root) > 0:
    first = root[0]
    raw = ET.tostring(first, encoding='unicode')
    print('\nFirst record (raw XML):')
    print(raw[:1500])

File: definitions_fitness.xml
Root tag: <definition-page>
Root's direct children tags: ['term-group', 'term-group', 'term-group', 'term-group', 'term-group']

First record (raw XML):
<term-group reference="National Heart, Lung, and Blood Institute" reference-url="https://www.nhlbi.nih.gov/health-topics/physical-activity-and-your-heart"><term>&gt;Activity Count</term><definition>&gt;Physical activity is any body movement that works your muscles and requires more energy than resting. Walking, running, dancing, swimming, yoga, and gardening are a few examples of physical activity.</definition></term-group>


**more checking stuff**

In [5]:
topic_files = sorted(DATA_DIR.glob('mplus_topics_*.xml'))
topic_files = [f for f in topic_files if 'compressed' not in f.name]
assert topic_files, 'No mplus_topics_*.xml file found -- check the unzip step in Section 1.'

sample_path = topic_files[0]
tree = ET.parse(sample_path)
root = tree.getroot()
topics = list(root)

print(f'File: {sample_path.name}')
print(f'Root tag: <{root.tag}>')
print(f'Number of health-topic records: {len(topics)}')

if topics:
    first = topics[0]
    print(f'First topic tag: <{first.tag}>')
    print(f'First topic attributes: {first.attrib}')
    print(f'First topic children tags: {[c.tag for c in list(first)]}')
    raw = ET.tostring(first, encoding='unicode')
    print('\nFirst record (raw XML, truncated):')
    print(raw[:2000])

File: mplus_topics_2026-09-29.xml
Root tag: <health-topics>
Number of health-topic records: 2033
First topic tag: <health-topic>
First topic attributes: {'meta-desc': 'A1C is a blood test for type 2 diabetes and prediabetes. It measures your average blood glucose, or blood sugar, levels over the past 3 months. Learn more.', 'title': 'A1C', 'url': 'https://medlineplus.gov/a1c.html', 'id': '6308', 'language': 'English', 'date-created': '12/22/2015'}
First topic children tags: ['also-called', 'also-called', 'also-called', 'full-summary', 'group', 'group', 'language-mapped-topic', 'mesh-heading', 'other-language', 'primary-institute', 'see-reference', 'see-reference', 'site', 'site', 'site', 'site', 'site', 'site', 'site', 'site']

First record (raw XML, truncated):
<health-topic meta-desc="A1C is a blood test for type 2 diabetes and prediabetes. It measures your average blood glucose, or blood sugar, levels over the past 3 months. Learn more." title="A1C" url="https://medlineplus.gov/a1c.

## Parsing, chunking, embedding, and storing in ChromaDB

In [6]:
def chunk_text(text: str, size: int = CHUNK_SIZE_CHARS, overlap: int = CHUNK_OVERLAP_CHARS) -> list[str]:
    """Simple sliding-window chunker. Swap for a sentence-aware chunker later
    if retrieval quality suggests it's cutting sentences awkwardly."""
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) <= size:
        return [text] if text else []
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += size - overlap
    return chunks


def parse_definitions(path: Path) -> list[dict]:
    tree = ET.parse(path)
    root = tree.getroot()
    records = []
    for group_el in root.iter('term-group'):
        term_el = group_el.find('term')
        definition_el = group_el.find('definition')
        if term_el is None or definition_el is None:
            continue
        term = (term_el.text or '').lstrip('>').strip()
        definition = (definition_el.text or '').lstrip('>').strip()
        if not term or not definition:
            continue
        records.append({
            'text': f'{term}: {definition}',
            'source_type': 'glossary',
            'title': term,
            'source_file': path.name,
            'reference': group_el.get('reference', ''),
            'reference_url': group_el.get('reference-url', ''),
        })
    return records

def strip_html(raw: str) -> str:
    """MedlinePlus full-summary content is HTML-escaped markup, not plain
    text. Unescape entities, then strip tags to get clean text for embedding."""
    unescaped = html_module.unescape(raw)
    return re.sub(r'<[^>]+>', ' ', unescaped)

import html as html_module
def parse_health_topics(path: Path) -> list[dict]:
    tree = ET.parse(path)
    root = tree.getroot()
    records = []
    for topic_el in root.iter('health-topic'):
        title = topic_el.get('title', '')
        url = topic_el.get('url', '')
        language = topic_el.get('language', '')
        if language and language.lower() != 'english':
            continue  # keep English-only for now; revisit for the Spanish stretch goal
        summary_el = topic_el.find('full-summary')
        raw_summary = (summary_el.text or '') if summary_el is not None else ''
        summary_text = strip_html(raw_summary).strip()
        if not title or not summary_text:
            continue
        records.append({
            'text': summary_text,
            'source_type': 'guideline',
            'title': title,
            'url': url,
            'source_file': path.name,
        })
    return records


print('Parsing functions defined.')

Parsing functions defined.


In [7]:
# Parse everything
records = []

for f in sorted(DATA_DIR.glob('definitions_*.xml')):
    recs = parse_definitions(f)
    print(f'  {f.name}: {len(recs)} glossary terms')
    records.extend(recs)

for f in sorted(DATA_DIR.glob('mplus_topics_*.xml')):
    if 'compressed' in f.name:
        continue
    recs = parse_health_topics(f)
    print(f'  {f.name}: {len(recs)} health topics')
    records.extend(recs)

print(f'\nTotal records parsed: {len(records)}')
assert records, 'No records parsed -- check Section 2 findings and fix the parser above.'

  definitions_fitness.xml: 16 glossary terms
  definitions_general_health.xml: 18 glossary terms
  definitions_minerals.xml: 16 glossary terms
  definitions_nutrition.xml: 30 glossary terms
  definitions_vitamins.xml: 17 glossary terms
  mplus_topics_2026-09-29.xml: 1014 health topics
  mplus_topics_2026-10-03.xml: 1015 health topics

Total records parsed: 2126


In [8]:
records = records[:300]  # just enough to prove the pipeline works
print(f'Trimmed to {len(records)} records for a faster first pass')

Trimmed to 300 records for a faster first pass


In [9]:
# Chunk
chunks = []
for rec in records:
    for i, chunk in enumerate(chunk_text(rec['text'])):
        chunks.append({
            'id': f"{rec['source_file']}::{rec['title']}::{i}",
            'text': chunk,
            'metadata': {
                'source_type': rec['source_type'],
                'title': rec['title'],
                'url': rec.get('url', ''),
                'source_file': rec['source_file'],
            },
        })

print(f'Total chunks after splitting: {len(chunks)}')

Total chunks after splitting: 423


In [10]:
import time as time_module

def embed_batch_with_retry(texts: list[str], max_retries: int = 5) -> list[list[float]]:
    for attempt in range(max_retries):
        try:
            response = client.models.embed_content(
                model=EMBED_MODEL,
                contents=texts,
                config=types.EmbedContentConfig(
                    task_type='RETRIEVAL_DOCUMENT',
                    output_dimensionality=768,
                ),
            )
            return [e.values for e in response.embeddings]
        except Exception as e:
            if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                wait = 40  # a little more than the 35s Google asked for
                print(f'    Rate limited, waiting {wait}s (attempt {attempt + 1}/{max_retries})...')
                time_module.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded on embedding call')


chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = chroma_client.get_or_create_collection(COLLECTION_NAME)

# Resumability: skip chunks that are already indexed, so a rate-limit hit
# earlier doesn't force you to re-embed everything from scratch
existing_ids = set(collection.get()['ids'])
remaining_chunks = [c for c in chunks if c['id'] not in existing_ids]
print(f'{len(existing_ids)} chunks already indexed, {len(remaining_chunks)} remaining')

for batch_start in range(0, len(remaining_chunks), EMBED_BATCH_SIZE):
    batch = remaining_chunks[batch_start:batch_start + EMBED_BATCH_SIZE]
    texts = [c['text'] for c in batch]
    embeddings = embed_batch_with_retry(texts)
    collection.add(
        ids=[c['id'] for c in batch],
        embeddings=embeddings,
        documents=texts,
        metadatas=[c['metadata'] for c in batch],
    )
    done = batch_start + len(batch)
    print(f'  {done}/{len(remaining_chunks)} chunks indexed')
    time.sleep(3.0)  # slower pace to stay under the free-tier 100/min limit

print(f"\nDone. Collection '{COLLECTION_NAME}' has {collection.count()} chunks.")

860 chunks already indexed, 0 remaining

Done. Collection 'medlineplus_glossary_and_guidelines' has 860 chunks.


## Test retrieval?? pray


In [11]:
def query_index(query: str, n_results: int = 5):
    response = client.models.embed_content(
        model=EMBED_MODEL,
        contents=[query],
        config=types.EmbedContentConfig(
            task_type='RETRIEVAL_QUERY',
            output_dimensionality=768,
        ),
    )
    query_embedding = response.embeddings[0].values

    results = collection.query(query_embeddings=[query_embedding], n_results=n_results)

    print(f'Query: {query}\n')
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0], results['metadatas'][0], results['distances'][0]
    )):
        print(f'--- Result {i + 1} (distance: {dist:.4f}) ---')
        print(f"Source: {meta['source_type']} | {meta['title']}")
        print(doc[:300])
        print()


query_index('what is a viral upper respiratory infection')

Query: what is a viral upper respiratory infection

--- Result 1 (distance: 0.2386) ---
Source: guideline | Antibiotics
 Flu Most cases of bronchitis What are the side effects of antibiotics? The side effects of antibiotics range from minor to very severe. Some of the common side effects include: Rash Nausea Diarrhea Yeast infections More serious side effects can include: C. diff infections , which cause diarrhea tha

--- Result 2 (distance: 0.2388) ---
Source: guideline | Antibiotics
ions , and E. coli . You may not need to take antibiotics for some bacterial infections. For example, you might not need them for many sinus infections or some ear infections . Taking antibiotics when they're not needed won't help you, and they can have side effects. Your health care provider can de

--- Result 3 (distance: 0.2469) ---
Source: guideline | Acute Bronchitis
Bronchitis is an inflammation of the bronchial tubes, the airways that carry air to your lungs. It causes a cough that often brings up 

In [12]:
print(collection.count())

860


## Summary

- Indexed **glossary** terms (fitness, general health, minerals, nutrition, vitamins definitions) and **guideline** summaries (all English health topics) into one ChromaDB collection.
- Each chunk is tagged with `source_type` (`glossary` vs `guideline`), `title`, and `url` (for guidelines) so the Simplifier agent can cite where a fact came from.

**things i found:**
- text-embedding-004 → gemini-embedding-001 (retired Jan 2026): changed this
- initial vector retrieval indexing built on 300/1111 records cuz the api kept crashing out
- Chunk boundaries sometimes cut mid-sentence — known v1 limitation